# 階段 8：text-to-SQL 微調

對應文件：`docs/08-stage-finetune.md`。這份 notebook 分三段：

1. **資料**（本機）：從階段 6 的產生器擴到數千題、依群組切分、防洩漏檢查、轉成 chat 格式
2. **Baseline**（本機）：大模型 API few-shot、開源小模型 zero-shot 與 self-correction
3. **QLoRA 訓練**（本機 GPU 或 Kaggle / Colab）：第 3 段可以整份上傳到 Colab 跑

**硬體**：6GB VRAM 的消費級 GPU（例如 RTX 2060）能用 4-bit QLoRA 微調 1.5B 到 3B；7B 到 8B 用 Kaggle 或 Colab 的 T4。Turing 架構（RTX 20 系列、T4）不支援 bf16，一律用 fp16。沒有 GPU 就整段在雲端做。

**授權**：選模型前先看 model card 的授權欄位。例如 Qwen2.5-Coder-3B-Instruct 是非商用的 qwen-research 授權，作品要公開展示就換別的尺寸或系列。

In [ ]:
import hashlib
import json
import random
import re
from collections import Counter
from pathlib import Path

ROOT = Path.cwd().parent  # kernel 的工作目錄是 notebooks/，往上一層才是專案根目錄
FT_DIR = ROOT / "data" / "finetune"

## 步驟 1：資料集與切分

產生器做出來的題目很像。如果按隨機列切分，測試題的「近親」（同樣板、同產品，只換門檻）會出現在訓練集，分數虛高。所以**依群組切分**：同一個「樣板 + 產品」只會落在一個集合，另外保留整個樣板只出現在測試集，量泛化能力。

先用階段 6 的 `evals/generate.py` 擴充樣板與問法，產出 3000 到 5000 題，每題都實際在資料庫執行一次，丟掉執行失敗或結果為空的題目。

In [ ]:
def norm_sql(sql: str) -> str:
    return re.sub(r"\s+", " ", sql.strip().rstrip(";")).lower()

def norm_q(q: str) -> str:
    return re.sub(r"\s+", "", q)

def group_of(case: dict) -> str:
    return f"{case['template']}|{case.get('params', {}).get('product', '')}"

def split(cases: list[dict], ratios=(0.8, 0.1, 0.1), holdout_templates: set[str] = frozenset()) -> dict[str, list]:
    """同一群組只會落在同一個集合；holdout_templates 整個樣板只進 test。用 hash 分配，重跑結果一樣。"""
    out = {"train": [], "val": [], "test": []}
    for c in cases:
        if c["template"] in holdout_templates:
            out["test"].append(c); continue
        h = int(hashlib.sha256(group_of(c).encode()).hexdigest(), 16) % 10_000 / 10_000
        name = "train" if h < ratios[0] else "val" if h < ratios[0] + ratios[1] else "test"
        out[name].append(c)
    return out

def leaks(a: list[dict], b: list[dict]) -> dict:
    """兩個集合之間重複的問題文字與正規化 SQL。"""
    return {"question": {norm_q(c["question"]) for c in a} & {norm_q(c["question"]) for c in b},
            "sql": {norm_sql(c["gold_sql"]) for c in a} & {norm_sql(c["gold_sql"]) for c in b}}

# 自我檢查
_cases = [{"template": t, "params": {"product": p, "cvss": s}, "question": f"{t}{p}{s}", "gold_sql": f"select {t} {p} {s}"}
          for t in ("A", "B", "C") for p in ("nginx", "openssl", "redis", "php") for s in (5, 7, 9)]
_s = split(_cases, holdout_templates={"C"})
assert sum(map(len, _s.values())) == len(_cases)
assert all(c in _s["test"] for c in _cases if c["template"] == "C")
_groups = {k: {group_of(c) for c in v} for k, v in _s.items()}
assert not (_groups["train"] & _groups["val"]) and not (_groups["train"] & _groups["test"])
assert not any(leaks(_s["train"], _s["test"]).values())
print({k: len(v) for k, v in _s.items()})

### 1.1 防洩漏檢查

階段 6 的固定評測集（200 題以上）與手寫刁鑽題，**一題都不能**出現在訓練集。

In [ ]:
# cases = [json.loads(l) for l in (FT_DIR / "all.jsonl").read_text().splitlines()]
# eval_set = [json.loads(l) for l in (ROOT / "evals" / "cases.jsonl").read_text().splitlines()]
# sets = split(cases, holdout_templates={"cwe_filter"})        # TODO：挑 1 到 2 個樣板當 holdout
# for name in ("val", "test"):
#     print(name, {k: len(v) for k, v in leaks(sets["train"], sets[name]).items()})
# assert not any(leaks(sets["train"], eval_set).values()), "評測集洩漏到訓練集"
# print({k: len(v) for k, v in sets.items()}, Counter(c["template"] for c in sets["train"]))

### 1.2 轉成 chat 格式

system prompt 放精簡的 schema 描述，**必須跟 agent 實際呼叫時用的一模一樣**，否則微調效果會在上線時消失。

In [ ]:
SYSTEM = ("你把漏洞問題轉成 DuckDB/PostgreSQL 唯讀 SQL。只輸出 SQL。\n"
          "資料表：cve(cve_id, published, description, cvss_score, severity, cwe, products)、"
          "kev(cve_id, vendor, product, date_added, due_date)、epss(cve_id, epss, percentile)")
# TODO：換成你階段 2 之後實際的 PostgreSQL schema，並從 src/vuln_agent 匯入同一個常數，不要複製貼上

def to_chat(case: dict) -> dict:
    return {"messages": [{"role": "system", "content": SYSTEM},
                         {"role": "user", "content": case["question"]},
                         {"role": "assistant", "content": case["gold_sql"]}]}

# FT_DIR.mkdir(parents=True, exist_ok=True)
# for name, rows in sets.items():
#     with open(FT_DIR / f"{name}.jsonl", "w") as f:
#         for c in rows:
#             f.write(json.dumps(to_chat(c), ensure_ascii=False) + "\n")

## 步驟 2：Baseline

沒有 baseline，就說不出微調到底提升了多少。全部用**同一份測試集、同一個判定**（執行結果等於標準答案，沿用階段 6 的 `same_result`）。

| 組別 | 做法 |
| --- | --- |
| 大模型 API few-shot | 從訓練集挑 5 到 8 題當範例 |
| 開源 zero-shot | Ollama 跑候選小模型，同樣的 prompt 不給範例 |
| 開源 + self-correction | SQL 執行失敗就把錯誤丟回模型重寫，最多兩次；**微調要贏過這組才算值得** |

Ollama 提供 OpenAI 相容 API，用 `openai` 套件、改 `base_url` 就能呼叫（`uv add openai`，端點以 Ollama 官方文件為準）；大模型 API 沿用 `vuln_agent.llm`。

In [ ]:
def generate_sql(question: str, provider: str, model: str, examples: list[dict] = ()) -> str:
    ...  # TODO：依 provider 呼叫 Claude API 或 OpenAI 相容端點，回傳 SQL 字串

def run_baseline(test: list[dict], provider: str, model: str, examples=(), self_correct: int = 0) -> dict:
    ...  # TODO：每題產生 SQL → 唯讀執行 → 跟 gold_result 比；記錄正確率、錯誤類型、p50 延遲、每題成本

把錯誤分類（語法錯、欄位名錯、條件漏掉、JOIN 錯、時間計算錯）存成表格，微調後對照哪些類型變少。

## 步驟 3：QLoRA 微調

這一段可以把 notebook 上傳到 Kaggle 或 Colab 跑，`data/finetune/*.jsonl` 一起上傳。Hugging Face token 放環境變數或平台的 secrets，不要寫進 notebook。

套件：Hugging Face Transformers + PEFT + TRL（或 Unsloth）。這幾個套件 API 變得快，**寫法以安裝版本的官方文件為準**，CUDA 版本也要對上。

**TODO**：寫訓練程式，重點設定：

- 底模以 4-bit 載入（bitsandbytes）
- LoRA rank 先試 16，target modules 先用 attention 的投影層
- **fp16，不要開 bf16**（Turing 架構不支援）
- gradient checkpointing、batch size 1 + gradient accumulation
- 序列長度依「system prompt + 問題 + SQL」的最大長度設定，不要盲目設大
- 每個 epoch 在驗證集跑 SQL 正確率挑 checkpoint，**不要只看 loss**
- 記錄 loss 曲線與 VRAM 峰值；OOM 時依序調低序列長度、rank、縮小 target modules，每次調整都記下來

In [ ]:
TRAIN_CONFIG = {
    "base_model": "",          # TODO：例如 Qwen2.5-Coder-1.5B-Instruct，授權先確認
    "load_in_4bit": True,
    "lora_r": 16,
    "lora_alpha": 32,
    "fp16": True,
    "bf16": False,
    "gradient_checkpointing": True,
    "per_device_train_batch_size": 1,
    "gradient_accumulation_steps": 16,
    "max_seq_length": 1024,
    "num_train_epochs": 3,
    "data_version": "",        # TODO：資料檔的 hash 或日期，確保可重現
}

# TODO：載入模型與資料 → 訓練 → 每個 epoch 評估 → 存 adapter
# 存檔時把 TRAIN_CONFIG 一起寫進 adapter 資料夾

## 步驟 4：量化比較

把最好的 adapter 合併後，分別做 4-bit、8-bit、fp16 推論（例如轉 GGUF 給 Ollama / llama.cpp，工具與指令以官方文件為準）。每個組合量 SQL 正確率、VRAM、p50 / p95 延遲、每秒 tokens，做成一張表：模型大小 × 量化方式 × 正確率 × VRAM × 延遲。

接著回到 `docs/08-stage-finetune.md` 的步驟 5，把模型切換接回 agent。